In [2]:
import sys
sys.path.append('..')
import requests
import os
import random
from helpers import *
from dotenv import load_dotenv

In [2]:
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

In [3]:
AVAILABLE_PLATFORMS = [
    "Netflix",
    "Amazon Prime Video",
    "JioHotstar",
    "Sony Liv",
    "Zee5",
    "YouTube"
]

In [4]:
def get_watch_providers(title, region="IN"):
    """
    Get streaming platforms for a movie in a region.
    Returns a list of platform names.
    """
    try:
        # Step A — get the movie's TMDB ID first
        clean_title = title.split("(")[0].strip()
        year = title.split("(")[1].replace(")", "").strip()
        
        search_url = "https://api.themoviedb.org/3/search/movie"
        params = {
            "api_key": TMDB_API_KEY,
            "query": clean_title,
            "year": year
        }
        response = requests.get(search_url, params=params, timeout=5)
        data = response.json()
        
        if not data["results"]:
            return []
        
        movie_id = data["results"][0]["id"]   # get the ID
        
        # Step B — call the watch providers endpoint
        provider_url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers"
        
        # Step B — call the watch providers endpoint
        prov_params = {"api_key": TMDB_API_KEY}
        
        prov_response = requests.get(provider_url, params=prov_params, timeout=5)
        prov_data = prov_response.json()
        
        # Step C — dig into results → region → flatrate
        results = prov_data.get("results", {})
        region_data = results.get(region, {})
        flatrate = region_data.get("flatrate", [])
        
        # Step D — extract just the platform names
        platforms = [p["provider_name"] for p in flatrate]
        
        return platforms
        
    except:
        return []

In [5]:
# Test with a popular movie
platforms = get_watch_providers("Toy Story (1995)")
print("Toy Story is on:", platforms)

Toy Story is on: ['JioHotstar', 'VI movies and tv']


In [6]:
def filter_by_platform(titles, user_platforms, region="IN"):
    """
    Keep only movies available on the user's platforms.
    Uses 'contains' matching to catch variations.
    """
    filtered = []
    
    for title in titles:
        movie_platforms = get_watch_providers(title, region)
        
        # Check if ANY user platform matches (contains)
        for user_plat in user_platforms:
            for movie_plat in movie_platforms:
                if user_plat in movie_plat:   #  'contains' check!
                    filtered.append(title)
                    break
            else:
                continue   # no match, keep checking
            break   # match found, stop
    
    return filtered

In [7]:
movies = ["Toy Story (1995)", "Jumanji (1995)", "The Dark Knight (2008)"]
user_platforms = ["Amazon Prime Video"]

result = filter_by_platform(movies, user_platforms)
print("On Amazon Prime Video:")
for m in result:
    print("  -", m)

On Amazon Prime Video:
  - Jumanji (1995)
  - The Dark Knight (2008)


In [8]:
jumanji_platforms = get_watch_providers("Jumanji (1995)")
print("Jumanji is on:", jumanji_platforms)

Jumanji is on: ['Amazon Prime Video', 'Lionsgate Play', 'Lionsgate Play Apple TV Channel', 'Lionsgate Play Amazon Channel', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']


In [9]:

def get_recommendations_with_platform(genre, user_platforms=None, n=5, user_id=None, region="IN"):
    """
    Get recommendations, optionally filter by platform.
    If user_platforms is empty → skip filtering.
    """
    # If no platforms chosen → skip filter
    if not user_platforms:
        return get_recommendations(genre=genre, n=n, user_id=user_id)
    
    # Otherwise → buffer, filter, return top n
    buffer_n = n * 4
    titles = get_recommendations(genre=genre, n=buffer_n, user_id=user_id)
    filtered = filter_by_platform(titles, user_platforms, region)
    return filtered[:n]

In [10]:
result = get_recommendations_with_platform(
    genre="Comedy",
    user_platforms=["JioHotstar", "Netflix"],
    n=5
)
print("Comedy movies you can watch:")
for movie in result:
    print("  →", movie)

Comedy movies you can watch:


In [11]:
# Step 1 — get the buffer (before filtering)
buffer_movies = get_recommendations(genre="Comedy", n=20, user_id=None)
print("Got", len(buffer_movies), "movies before filtering")

# Step 2 — check how many survive the filter
filtered = filter_by_platform(buffer_movies, ["JioHotstar", "Netflix"])
print("After filtering:", len(filtered), "movies survived")
print()

# Step 3 — see which survived
for m in filtered:
    print("  →", m)

Got 10 movies before filtering
After filtering: 0 movies survived



In [12]:
# Check what TMDB calls each platform
test_movies = [
    "Toy Story (1995)",
    "Jumanji (1995)",
    "The Dark Knight (2008)",
    "Inception (2010)",
]

for movie in test_movies:
    platforms = get_watch_providers(movie)
    print(f"{movie}:")
    print(f"  {platforms}\n")

Toy Story (1995):
  ['JioHotstar', 'VI movies and tv']

Jumanji (1995):
  ['Amazon Prime Video', 'Lionsgate Play', 'Lionsgate Play Apple TV Channel', 'Lionsgate Play Amazon Channel', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']

The Dark Knight (2008):
  ['Amazon Prime Video', 'JioHotstar', 'Amazon Prime Video with Ads']

Inception (2010):
  ['Amazon Prime Video', 'JioHotstar', 'Amazon Prime Video with Ads']



In [13]:


# Quick test
platforms = get_watch_providers("Toy Story (1995)")
print("Works from helpers:", platforms)

Works from helpers: ['JioHotstar', 'VI movies and tv']


In [14]:

buffer = get_recommendations(genre="Animation", n=24, user_id=1)
print("Movies BEFORE filter:", len(buffer))

# Filter by Netflix
filtered = filter_by_platform(buffer, ["Netflix"])
print("Movies AFTER Netflix filter:", len(filtered))
print()
for m in filtered:
    print("  →", m)

Movies BEFORE filter: 20
Movies AFTER Netflix filter: 2

  → Nausicaä of the Valley of the Wind (Kaze no tani no Naushika) (1984)
  → Laputa: Castle in the Sky (Tenkû no shiro Rapyuta) (1986)


In [15]:


# Get Animation movies (like your test)
buffer = get_recommendations(genre="Animation", n=24, user_id=1)
print("Movies BEFORE filter:", len(buffer))

# Filter by Netflix
filtered = filter_by_platform(buffer, ["Netflix"])
print("Movies AFTER Netflix filter:", len(filtered))
print()
for m in filtered:
    print("  →", m)

Movies BEFORE filter: 20
Movies AFTER Netflix filter: 3

  → Kung Fu Panda 3 (2016)
  → Nausicaä of the Valley of the Wind (Kaze no tani no Naushika) (1984)
  → Laputa: Castle in the Sky (Tenkû no shiro Rapyuta) (1986)


In [16]:
for movie in ["Toy Story (1995)", "Shrek (2001)"]:
    print(movie, get_watch_providers(movie))

Toy Story (1995) ['JioHotstar', 'VI movies and tv']
Shrek (2001) ['Amazon Prime Video', 'Amazon Prime Video with Ads']


In [17]:

def get_genre_map():
    """
    Build a dictionary that translates genre NAMES → TMDB genre IDs.
    e.g. {"Animation": 16, "Action": 28, ...}
    """
    url = "https://api.themoviedb.org/3/genre/movie/list"
    params = {"api_key": TMDB_API_KEY, "language": "en-US"}
    response = requests.get(url, params=params, timeout=5).json()

    genre_map = {}
    for g in response["genres"]:
        genre_map[g["name"]] = g["id"]     # "Animation" → 16

    return genre_map

In [18]:
GENRE_MAP = get_genre_map()
print(GENRE_MAP)

{'Action': 28, 'Adventure': 12, 'Animation': 16, 'Comedy': 35, 'Crime': 80, 'Documentary': 99, 'Drama': 18, 'Family': 10751, 'Fantasy': 14, 'History': 36, 'Horror': 27, 'Music': 10402, 'Mystery': 9648, 'Romance': 10749, 'Science Fiction': 878, 'TV Movie': 10770, 'Thriller': 53, 'War': 10752, 'Western': 37}


In [19]:

LLM_TO_TMDB_GENRE = {
    "Sci-Fi":   "Science Fiction",
    "Children": "Family",
    "Musical":  "Music",
}

def get_genre_id(llm_genre):
    tmdb_name = LLM_TO_TMDB_GENRE.get(llm_genre, llm_genre)   # translate, or keep as-is
    drama_id  = GENRE_MAP["Drama"]
    genre_id  = GENRE_MAP.get(tmdb_name, drama_id)            # find ID, or fall back to Drama
    return genre_id

In [20]:
print(get_genre_id("Action"))      # expect 28  (passes straight through)
print(get_genre_id("Sci-Fi"))      # expect 878 (translated → Science Fiction)
print(get_genre_id("Film-Noir"))   # expect 18  (unknown → falls back to Drama)
print(get_genre_id("Comedy"))      # expect 35  (passes straight through)

28
878
18
35


In [21]:
PLATFORM_IDS = {
    "Netflix": 8,
    "Amazon Prime Video": 119,
    "JioHotstar": 2336,
    "Zee5": 232,
    "Sony Liv": 237,
    "YouTube": 192,
}

In [22]:
def discover_movies_by_genre(genre_id, n=20, provider_ids=None):
    url = "https://api.themoviedb.org/3/discover/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "with_genres": genre_id,
        "sort_by": "popularity.desc",
        "watch_region": "IN",
        "vote_count.gte": 100,
        "page": random.randint(1, 3),
    }
    # NEW — if platforms given, ask TMDB for movies ON those platforms
    if provider_ids:
        params["with_watch_providers"] = "|".join(str(pid) for pid in provider_ids)
        params["watch_region"] = "IN"
        params["with_watch_monetization_types"] = "flatrate"

    response = requests.get(url, params=params, timeout=5).json()
    return response.get("results", [])[:n]

In [23]:
movies = discover_movies_by_genre(878, n=5)   # 878 = Sci-Fi

print("How many movies:", len(movies))
print()

# Peek at the FIRST movie's shape
first = movies[0]
print("Title:", first["title"])
print("ID:", first["id"])
print("Overview:", first["overview"][:80], "...")
print("Poster:", first["poster_path"])

How many movies: 5

Title: War Machine
ID: 1265609
Overview: On one last grueling mission during Army Ranger training, a combat engineer must ...
Poster: /rFhKkXhk7ClU03jQ5rHIApJDwev.jpg


In [24]:
def get_providers_by_id(movie_id, region="IN"):
    try:
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers"
        params = {"api_key": TMDB_API_KEY}
        response = requests.get(url, params=params, timeout=10).json()
        results = response.get("results", {})
        region_data = results.get(region, {})
        flatrate = region_data.get("flatrate", [])
        return [p["provider_name"] for p in flatrate]
    except:
        return []

In [25]:
# Get fresh sci-fi movies, then check the FIRST one's platforms
movies = discover_movies_by_genre(878, n=5)

first = movies[0]
print("Movie:", first["title"])
print("ID:", first["id"])

platforms = get_providers_by_id(first["id"])
print("Platforms:", platforms)

Movie: Disclosure Day
ID: 1275779
Platforms: []


In [26]:
movies = discover_movies_by_genre(878, n=5)

for m in movies:
    platforms = get_providers_by_id(m["id"])
    print(m["title"], "→", platforms)

War Machine → ['Netflix']
Masters of the Universe → []
Real Steel → []
Minions: The Rise of Gru → ['JioHotstar']
Avatar → ['JioHotstar']


In [27]:
def guest_recommendations_with_platform(genre_name, user_platforms=None, n=5, region="IN"):
    genre_id = get_genre_id(genre_name)

    # NEW — convert platform names → provider IDs
    provider_ids = None
    if user_platforms:
        provider_ids = [PLATFORM_IDS[p] for p in user_platforms if p in PLATFORM_IDS]

    # Pass provider_ids into discover
    buffer = discover_movies_by_genre(genre_id, n=n * 4, provider_ids=provider_ids)

    survivors = []
    for movie in buffer:
        platforms = get_providers_by_id(movie["id"], region)
        if not platforms:
            continue
        if not user_platforms:
            movie["platforms"] = platforms
            survivors.append(movie)
        else:
            for user_plat in user_platforms:
                for movie_plat in platforms:
                    if user_plat.lower() in movie_plat.lower():
                        movie["platforms"] = platforms
                        survivors.append(movie)
                        break
                else:
                    continue
                break
        if len(survivors) >= n:
            break
    return survivors

In [28]:
# Test A — a genre that goes through TRANSLATION (Children → Family)
results = guest_recommendations_with_platform(
    genre_name="Children",
    user_platforms=["Netflix", "Amazon Prime Video", "JioHotstar"],
    n=5
)
print("CHILDREN →", len(results), "found")
for m in results:
    print(" ", m["title"], "→", m["platforms"])

print()

# Test B — a normal pass-through genre, streaming-poorer maybe (Horror)
results = guest_recommendations_with_platform(
    genre_name="Horror",
    user_platforms=["Netflix", "Amazon Prime Video", "JioHotstar"],
    n=5
)
print("HORROR →", len(results), "found")
for m in results:
    print(" ", m["title"], "→", m["platforms"])

CHILDREN → 5 found
  Swapped → ['Netflix']
  The Sheep Detectives → ['Amazon Prime Video', 'Amazon Prime Video with Ads']
  Hoppers → ['JioHotstar']
  Zootopia 2 → ['JioHotstar']
  Moana 2 → ['JioHotstar']

HORROR → 5 found
  Dolly → ['Amazon Prime Video', 'Amazon Prime Video with Ads']
  Send Help → ['JioHotstar']
  Evil Dead Rise → ['JioHotstar']
  The Conjuring: Last Rites → ['JioHotstar']
  28 Years Later: The Bone Temple → ['Netflix']


In [29]:
# Check what TMDB actually calls these platforms
movies = discover_movies_by_genre(get_genre_id("Drama"), n=24)
all_platforms = set()
for m in movies:
    for p in get_providers_by_id(m["id"]):
        all_platforms.add(p)

print(sorted(all_platforms))

['Amazon Prime Video', 'Amazon Prime Video with Ads', 'JioHotstar', 'Netflix', 'VI movies and tv']


In [30]:
all_platforms = set()

for g in ["Drama", "Animation", "Action", "Comedy", "Romance", "Thriller"]:
    movies = discover_movies_by_genre(get_genre_id(g), n=20)
    for m in movies:
        for p in get_providers_by_id(m["id"]):
            all_platforms.add(p)

print(sorted(all_platforms))

['Amazon Prime Video', 'Amazon Prime Video with Ads', 'Apple TV', 'Apple TV Amazon Channel', 'Crunchyroll', 'Crunchyroll Amazon Channel', 'FilmBox+', 'JioHotstar', 'Lionsgate Play', 'Lionsgate Play Amazon Channel', 'Lionsgate Play Apple TV Channel', 'MGM Plus Amazon Channel', 'Netflix', 'Sony Liv', 'Sony Pictures Amazon Channel', 'VI movies and tv']


In [31]:

params = {"api_key": TMDB_API_KEY, "watch_region": "IN"}
data = requests.get(url, params=params).json()

for p in data["results"]:
    name = p["provider_name"]
    if "zee" in name.lower() or "youtube" in name.lower() or "sony" in name.lower():
        print(p["provider_id"], "→", name)

NameError: name 'url' is not defined

In [ ]:
wanted = ["Netflix", "Amazon Prime Video", "JioHotstar", "Zee5", "Sony Liv", "YouTube"]
for p in data["results"]:
    if p["provider_name"] in wanted:
        print(p["provider_id"], "→", p["provider_name"])

8 → Netflix
119 → Amazon Prime Video
2336 → JioHotstar
232 → Zee5
237 → Sony Liv
192 → YouTube


In [ ]:
movies = discover_movies_by_genre(get_genre_id("Drama"), n=10, provider_ids=[232])
print("Count:", len(movies))
for m in movies:
    print(m["title"])

Count: 10
Top Gun: Maverick
Saving Private Ryan
Oppenheimer
Schindler's List
Gladiator
Fifty Shades of Grey
Fifty Shades Freed
The Pursuit of Happyness
Inglourious Basterds
The Truman Show


In [ ]:
# Test A — Zee5 WITH the vote gate (current behavior)
movies = discover_movies_by_genre(get_genre_id("Drama"), n=20, provider_ids=[232])
print("Zee5 Drama (with gate):", len(movies))

# Test B — Zee5 WITHOUT the vote gate
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "api_key": TMDB_API_KEY,
    "with_genres": get_genre_id("Drama"),
    "with_watch_providers": 232,
    "watch_region": "IN",
    "sort_by": "popularity.desc",
    # NO vote_count.gte
}
data = requests.get(url, params=params).json()
print("Zee5 Drama (no gate):", len(data.get("results", [])))
for m in data["results"][:8]:
    print(" ", m["title"], "| votes:", m.get("vote_count"))

Zee5 Drama (with gate): 20
Zee5 Drama (no gate): 20
  Top Gun: Maverick | votes: 11144
  Saving Private Ryan | votes: 17396
  Oppenheimer | votes: 11964
  Schindler's List | votes: 17585
  Gladiator | votes: 21056
  Fifty Shades of Grey | votes: 12436
  Fifty Shades Freed | votes: 8378
  The Pursuit of Happyness | votes: 10664


In [ ]:

data = requests.get(url, params={"api_key": TMDB_API_KEY}).json()
print(data["results"]["IN"])   # print the FULL India block — flatrate, rent, buy, ads

{'link': 'https://www.themoviedb.org/movie/361743-top-gun-maverick/watch?locale=IN', 'rent': [{'logo_path': '/SPnB1qiCkYfirS2it3hZORwGVn.jpg', 'provider_id': 2, 'provider_name': 'Apple TV Store', 'display_priority': 5}, {'logo_path': '/gP67NRy1ShUJilrzMsbOmEmdmcv.jpg', 'provider_id': 232, 'provider_name': 'Zee5', 'display_priority': 7}, {'logo_path': '/8z7rC8uIDaTM91X0ZfkRf04ydj2.jpg', 'provider_id': 3, 'provider_name': 'Google Play Movies', 'display_priority': 8}, {'logo_path': '/pTnn5JwWr4p3pG8H6VrpiQo7Vs0.jpg', 'provider_id': 192, 'provider_name': 'YouTube', 'display_priority': 11}, {'logo_path': '/qR6FKvnPBx2O37FDg8PNM7efwF3.jpg', 'provider_id': 10, 'provider_name': 'Amazon Video', 'display_priority': 39}], 'flatrate': [{'logo_path': '/pvske1MyAoymrs5bguRfVqYiM9a.jpg', 'provider_id': 119, 'provider_name': 'Amazon Prime Video', 'display_priority': 1}, {'logo_path': '/kVqjgpcwvDJOhCupjcLzwwtOp52.jpg', 'provider_id': 2336, 'provider_name': 'JioHotstar', 'display_priority': 3}, {'logo_

In [ ]:
movies = discover_movies_by_genre(get_genre_id("Drama"), n=20, provider_ids=[237])
count = 0
for m in movies:
    plats = get_providers_by_id(m["id"])
    if any("sony liv" in p.lower() for p in plats):
        count += 1
        print("✅", m["title"], "→", plats)
print("Sony Liv flatrate hits:", count, "/", len(movies))

✅ Blade Runner 2049 → ['Amazon Prime Video', 'JioHotstar', 'Sony Liv', 'VI movies and tv', 'Amazon Prime Video with Ads']
✅ Your Name. → ['Crunchyroll', 'Sony Liv', 'Crunchyroll Amazon Channel']
✅ Once Upon a Time... in Hollywood → ['Amazon Prime Video', 'Sony Liv', 'VI movies and tv', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']
✅ Call Me by Your Name → ['Sony Liv', 'Sony Pictures Amazon Channel']
✅ A Silent Voice: The Movie → ['Crunchyroll', 'Sony Liv', 'Crunchyroll Amazon Channel']
✅ The Karate Kid → ['Amazon Prime Video', 'JioHotstar', 'Sony Liv', 'Sony Pictures Amazon Channel', 'Amazon Prime Video with Ads']
✅ The Whale → ['Sony Liv']
✅ Suzume → ['Crunchyroll', 'Sony Liv', 'Crunchyroll Amazon Channel']
✅ Close Encounters of the Third Kind → ['Sony Liv', 'Sony Pictures Amazon Channel']
✅ The Patriot → ['Sony Liv', 'Sony Pictures Amazon Channel']
✅ Little Women → ['Amazon Prime Video', 'Sony Liv', 'VI movies and tv', 'Sony Pictures Amazon Channel', 'Amazon Prime Vi

In [34]:
rejected, avoided = get_user_feedback(1)
print("REJECTED:", [r for r in rejected if "Boot" in r])

titles = get_recommendations(genre="Action", n=12, user_id=1)
print("IN TITLES:", [t for t in titles if "Boot" in t])

REJECTED: []
IN TITLES: []


In [35]:
print(FEEDBACK_FILE)
df = pd.read_csv(FEEDBACK_FILE)
print(df.tail(3))

d:\StreamWise\data\feedback.csv
   user_id                       movie_title  action           reason  \
13     129  Anvil! The Story of Anvil (2008)  reject        storyline   
14       1      Boot, Das (Boat, The) (1981)  reject  already_watched   
15       1      Boot, Das (Boat, The) (1981)  reject  already_watched   

      genre                   timestamp  
13  Musical  2026-07-15T16:03:36.360512  
14   Action  2026-07-16T10:39:48.327669  
15   Action  2026-07-16T10:40:46.380484  


In [36]:
rejected, avoided = get_user_feedback(1)
print("REJECTED titles:", rejected)

REJECTED titles: set()


In [37]:
df = pd.read_csv(FEEDBACK_FILE)
print(df["user_id"].dtype)              # likely 'object' (string)
print(df["user_id"].unique())           # likely ['1' '129' ... 'guest']

object
['1' '250' 'guest' '129']


In [ ]:
print(get_user_feedback(1))


({'Crumb (1994)', 'Ox-Bow Incident, The (1943)', 'Boot, Das (Boat, The) (1981)', 'Top Gun', 'Inception', 'Philadelphia Story, The (1940)', 'Meatballs III (1987)', 'Toy Story 3 (2010)', 'Real McCoy, The (1993)'}, {'Comedy', 'Sci-Fi', 'Action'})
